# Linear observables of the unitary QELM: distributions against the dataset size

Experiment `experiments/QELM_Z/exp_QELM_Z.py`, the simple QELM, with no measurement at all (`p = 0`, issue #191). The reservoir is a disordered TFIM `H = h sum_i Z_i + sum_{i<j} J_ij X_i X_j` on all `N` qubits, with a new disorder `J_ij` for every sample. The initial state entangles the ancilla `R` (qubit 0) with the charge of the `Ns = N - 1` system qubits, `(|0>_R |Q1>_S + |1>_R |Q2>_S) / sqrt(2)` with `Q1 = Ns // 2` and `Q2 = Q1 - 1`. Each sample evolves unitarily for the time `t`, `|psi> = exp(-i H t) |psi_0>`, and the features are the **linear observables** `<O_k> = <psi| O_k |psi>` of the system qubits, for `O = Z, Y, X`, plus a bias. The target of the ridge readout is the ancilla `<Z_0>`.

Over a dataset, each `<O_k>` is a random variable (random disorder, hence random unitary) in `[-1, 1]`. This notebook:

1. generates independent datasets of increasing size `n_samples` (100, 500, 2000, 5000 by default), for two fields `h`;
2. isolates the linear observables `<X_k>, <Y_k>, <Z_k>` (`QELM.evaluation.feature_distribution.linear_observables`);
3. fits the ridge readout on them only, all three together and each alone, to measure their baseline predictive power;
4. superimposes the histograms (and a kernel density estimate) of one observable, `<O_{Ns/2}>`, for the different `n_samples` (`QELM.utils.plots.draw_histograms`);
5. quantifies the convergence of the empirical distribution with `n_samples` (law of large numbers) and analyzes its final shape against a Gaussian and against Haar-random states.

**What to expect from the symmetries.** Every term of `H` conserves the parity of the number of excitations, `P = prod_i Z_i`: `Z_i` is diagonal and `X_i X_j` flips two spins. The initial state has a definite parity: the branch `|0>_R |Q1>` holds `Q1` excitations and the branch `|1>_R |Q2>` holds `1 + Q2 = Q1`. A single `X_k` or `Y_k` flips one spin, so it maps the parity sector of the state to the other one, and `<psi| X_k |psi> = <psi| Y_k |psi> = 0` **exactly**, for every sample and every `h`. Only `<Z_k>` carries information; its distribution is the object of this notebook.

Two references for `<Z_k>`, both Haar-random pure states (the limit of a fully scrambling unitary) drawn in the space the dynamics can reach (`QELM.evaluation.feature_distribution.haar_pdf`):

- the **parity sector**, of dimension `D = 2^(N-1)`, exact for any `h`: `<Z_k> = 2 B - 1` with `B ~ Beta(D/2, D/2)`, a symmetric law of mean 0 and standard deviation `1 / sqrt(D + 1)`;
- the **charge sector** of `Q = Q1` excitations among the `N` qubits, of dimension `C(N, Q)`, the relevant one when `h >> J` nearly conserves the charge: `B ~ Beta(C(N-1, Q), C(N, Q) - C(N-1, Q))`, of mean `1 - 2 Q / N`, skewed when `Q != N / 2`.

In [ ]:
import os
from pathlib import Path

# Each sample is a small sparse evolution: a single thread is fastest
# (notebook 06 measured it), and must be set before NumPy is imported.
os.environ.setdefault("OMP_NUM_THREADS", "1")

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_z_linear_notebook.py` does for a small run). `N` is the total number of qubits, ancilla included; `H_VALUES` are the transverse fields `TFIM.h` in units of `TFIM.J = 1`; `N_SAMPLES` are the dataset sizes. The dataset of the `k`-th size uses the seed `SEED + k`, so the datasets are independent: the distance between two of them measures the sampling noise only. `SITE` is the system qubit whose observables are plotted, `Ns // 2` by default (chain site `SITE + 1`, after the ancilla). Each run goes to this notebook's own folder, `outputs/QELM_Z_linear/<date_time>/`; `RUN = False` reads back the latest one without simulating again.

With the defaults, `N = 6` and `h = 10 J, 1 J`, the 2 x 7600 samples take about 20 minutes (0.1 s per sample at `h = 10 J`, fewer at `h = J`).

In [ ]:
N = int(os.environ.get("N", "6"))  # total qubits, ancilla included
H_VALUES = [float(h) for h in os.environ.get("H_VALUES", "10 1").split()]
N_SAMPLES = [int(n) for n in os.environ.get("N_SAMPLES", "100 500 2000 5000").split()]
SEED = int(os.environ.get("SEED", "42"))
SITE = int(os.environ.get("SITE", str((N - 1) // 2)))  # system qubit, 0 .. Ns-1
BINS = int(os.environ.get("BINS", "40"))
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"
LABELS = ["Z", "Y", "X"]  # experiment.observables, the default of exp_QELM_Z

## Generate the datasets

One run of `exp_QELM_Z.run` per field `h` and size `n_samples`, in this process, with `samples.L = n_samples` (70 % train, 30 % test, as in the config). Each run is saved in its own subfolder `h<h>_n<n_samples>/` of the sweep folder, with `exp_QELM_Z.npz` (the features `X`, bias column included, the target `Y`, the test predictions), `metrics.json` and `resolved_config.json`. The cell then reads the runs back from that folder, whether it just wrote them or not.

In [ ]:
import importlib.util
import json

import numpy as np

from QELM import load_config
from QELM.utils import latest_sweep, new_sweep_dir, save_config, save_results

spec = importlib.util.spec_from_file_location(
    "exp_QELM_Z", Path("experiments/QELM_Z/exp_QELM_Z.py")
)
assert spec is not None and spec.loader is not None
exp_QELM_Z = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp_QELM_Z)

EXPERIMENT = "QELM_Z_linear"  # this notebook's own output folder (issue #151)


def overrides(h: float, k: int, n: int) -> list[str]:
    return [
        "models=TFIM",
        f"N={N}",
        f"TFIM.h={h}",
        f"seed={SEED + k}",
        f"samples.L={n}",
        f"+experiment.observables=[{','.join(LABELS)}]",
    ]


if RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    for h in H_VALUES:
        for k, n in enumerate(N_SAMPLES):
            print(f"h = {h:g} J, n_samples = {n}")
            cfg = load_config(overrides(h, k, n))
            out = exp_QELM_Z.run(cfg)
            run_dir = sweep_dir / f"h{h:g}_n{n}"
            save_results(out, run_dir, "exp_QELM_Z")
            save_config(cfg, run_dir)
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

# Read back every run of the sweep folder.
runs = []
for run_dir in sorted(p for p in sweep_dir.iterdir() if p.is_dir()):
    cfg = json.loads((run_dir / "resolved_config.json").read_text())
    data = np.load(run_dir / "exp_QELM_Z.npz")
    runs.append(
        {
            "h": float(cfg["TFIM"]["h"]),
            "n_samples": int(cfg["samples"]["L"]),
            "seed": int(cfg["seed"]),
            "N": int(cfg["N"]),
            "Ns": int(cfg["Ns"]),
            "train_size": int(cfg["samples"]["train_size"]),
            "X": data["X"],
            "Y": data["Y"],
            "metrics": json.loads((run_dir / "metrics.json").read_text()),
            "cfg": cfg,
        }
    )
runs.sort(key=lambda r: (-r["h"], r["n_samples"]))
hs = sorted({r["h"] for r in runs}, reverse=True)
sizes = sorted({r["n_samples"] for r in runs})
Ns = runs[0]["Ns"]
assert all(r["N"] == N for r in runs), "the sweep was run with another N"
print(f"{len(runs)} runs: h = {hs}, n_samples = {sizes}, Ns = {Ns}")

## Isolate the linear observables

`linear_observables` splits the feature matrix of each run into one block of shape `(n_samples, Ns)` per Pauli and drops the bias column. The largest `|<X_k>|` and `|<Y_k>|` over all samples and sites check the parity argument above: they must be zero up to the rounding of the evolution.

In [ ]:
from QELM.evaluation.feature_distribution import linear_observables

for r in runs:
    r["obs"] = linear_observables(r["X"], LABELS, Ns)

for h in hs:
    rows_h = [r for r in runs if r["h"] == h]
    largest = {
        label: max(float(np.max(np.abs(r["obs"][label]))) for r in rows_h)
        for label in LABELS
    }
    print(
        f"h = {h:g} J: max |<Z_k>| = {largest['Z']:.3f}, "
        f"max |<Y_k>| = {largest['Y']:.1e}, max |<X_k>| = {largest['X']:.1e}"
    )

## Baseline predictive power of the linear features

The ridge readout of `exp_QELM_Z` (no intercept, hence the bias column; `training.alpha` of `conf/training/regression/ridge.yaml`) predicts the ancilla `<Z_0>` from all the linear features, `[<Z_k>, <Y_k>, <X_k>, 1]`: its test R2 and NMSE are in `metrics.json`. The same readout is refitted here on each Pauli alone, with the same train/test split (the first `train_size` samples train). A feature set that is identically 0 leaves only the bias: the readout predicts the mean of the train set, and R2 is at most 0.

Saved as `ridge_vs_n_samples.png` and `.json` in the sweep folder.

In [ ]:
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from QELM.evaluation.metrics import R2
from QELM.training import RidgeReadout
from QELM.utils.plots import color_map, save_figure, subplots

FEATURE_SETS = {"Z, Y, X": LABELS, "Z": ["Z"], "Y": ["Y"], "X": ["X"]}


def ridge_r2(r: dict, labels: list[str]) -> float:
    features = np.hstack([r["obs"][label] for label in labels])
    features = np.hstack([features, np.ones((features.shape[0], 1))])
    train = r["train_size"]
    solver = RidgeReadout.from_config(OmegaConf.create(r["cfg"]))
    solver.fit(features[:train], r["Y"][:train])
    return R2(r["Y"][train:], solver.predict(features[train:]))


ridge_table = []
for r in runs:
    row = {"h": r["h"], "n_samples": r["n_samples"]}
    for name, labels in FEATURE_SETS.items():
        row[f"r2_{name}"] = ridge_r2(r, labels)
    # Refitting on all the features reproduces the run's own score.
    assert np.isclose(row["r2_Z, Y, X"], r["metrics"]["r2"])
    ridge_table.append(row)

print(f"{'h':>5} {'n':>6} " + " ".join(f"{'R2 ' + name:>11}" for name in FEATURE_SETS))
for row in ridge_table:
    print(
        f"{row['h']:>5g} {row['n_samples']:>6} "
        + " ".join(f"{row[f'r2_{name}']:>11.4f}" for name in FEATURE_SETS)
    )

fig_r2, axes_r2 = subplots(1, len(hs), figsize=(6.5 * len(hs), 5), squeeze=False)
set_colors = color_map(FEATURE_SETS)
for h, ax in zip(hs, axes_r2[0], strict=True):
    rows_h = [row for row in ridge_table if row["h"] == h]
    for name in FEATURE_SETS:
        ax.plot(
            [row["n_samples"] for row in rows_h],
            [row[f"r2_{name}"] for row in rows_h],
            "o-",
            color=set_colors[name],
            label=f"<{name}>" if len(name) == 1 else "<Z>, <Y>, <X>",
        )
    ax.axhline(0.0, color="gray", linestyle=":")
    ax.set(
        xscale="log",
        xlabel="n_samples",
        ylabel="test R2 of <Z_0>",
        title=f"ridge on linear observables, h = {h:g} J",
    )
    ax.legend()
fig_r2.suptitle(f"Unitary QELM, N = {N}: baseline of the linear features")
ridge_path = save_figure(
    fig_r2, sweep_dir, "ridge_vs_n_samples", data={"table": ridge_table}
)
print(f"figure: {ridge_path}")
plt.show()

## Superimposed histograms of `<O_SITE>`

One row per field, one column per Pauli. In each panel, the density histogram of the observable on system qubit `SITE` for every `n_samples`, with common bins and a Gaussian KDE line on top. On the `<Z>` panels, the dashed black line is the Haar law in the parity sector and the dotted one the Haar law in the charge sector; the thin gray line is the Gaussian of the mean and standard deviation of the largest dataset. The `<Y>` and `<X>` panels hold a single bar at 0, the same for every size.

Saved as `histograms_linear_observables.png` and `.json`.

In [ ]:
from math import comb

from QELM.embedding import AncillaChargeSectorEmbedding
from QELM.evaluation.feature_distribution import haar_pdf
from QELM.utils.plots import common_edges, draw_histograms

sector = AncillaChargeSectorEmbedding.from_config(load_config(overrides(hs[0], 0, 10)))
Q = sector.Q1  # excitations in both branches: Q1 = 1 + Q2
assert sector.Q2 + 1 == Q
D_parity = 2 ** (N - 1)
D_charge, plus_charge = comb(N, Q), comb(N - 1, Q)
print(
    f"parity sector: D = {D_parity}; charge sector Q = {Q}: D = {D_charge}, "
    f"mean <Z_k> = 1 - 2 Q / N = {1 - 2 * Q / N:.3f}"
)

size_labels = {n: f"n = {n}" for n in sizes}
size_colors = color_map(size_labels.values())
fig_h, axes_h = subplots(
    len(hs), len(LABELS), figsize=(6.5 * len(LABELS), 5 * len(hs)), squeeze=False
)
z_grid = np.linspace(-1, 1, 801)
for h, row_axes in zip(hs, axes_h, strict=True):
    rows_h = {r["n_samples"]: r for r in runs if r["h"] == h}
    for label, ax in zip(LABELS, row_axes, strict=True):
        samples = {size_labels[n]: rows_h[n]["obs"][label][:, SITE] for n in sizes}
        edges = common_edges(list(samples.values()), BINS)
        handles = draw_histograms(ax, samples, size_colors, edges=edges)
        if label == "Z":
            largest = samples[size_labels[sizes[-1]]]
            mu, sigma = float(np.mean(largest)), float(np.std(largest, ddof=1))
            (gauss,) = ax.plot(
                z_grid,
                np.exp(-((z_grid - mu) ** 2) / (2 * sigma**2))
                / (sigma * np.sqrt(2 * np.pi)),
                color="gray",
                lw=1,
                label="Gaussian fit (largest n)",
            )
            (par,) = ax.plot(
                z_grid,
                haar_pdf(z_grid, D_parity),
                "k--",
                lw=1.5,
                label=f"Haar, parity sector (D = {D_parity})",
            )
            (chg,) = ax.plot(
                z_grid,
                haar_pdf(z_grid, D_charge, plus_charge),
                "k:",
                lw=1.5,
                label=f"Haar, charge sector Q = {Q} (D = {D_charge})",
            )
            handles += [gauss, par, chg]
            ax.set_xlim(-1.02, 1.02)
        ax.set(
            xlabel=f"<{label}_{SITE + 1}>",
            ylabel="density",
            title=f"<{label}> on system qubit {SITE}, h = {h:g} J",
        )
        ax.legend(handles=handles, fontsize=9)
fig_h.suptitle(f"Unitary QELM, N = {N}: linear observables against n_samples")

## Convergence with `n_samples` and final shape of `<Z_SITE>`

For each field, `convergence_table` gives the moments of every dataset and its two-sample Kolmogorov-Smirnov distance to the largest one. Two independent samples of the same law of sizes `n` and `m` lie at a distance of order `sqrt((n + m) / (n m))` (dashed), about `1 / sqrt(n)` for `n << m`: the empirical law converges at that rate (Glivenko-Cantelli), and a distance far above it would mean that the datasets do not sample the same law. The other panels show the mean (with its standard error) and the standard deviation against `n_samples`.

The shape table compares the largest dataset, for every system qubit, with the Gaussian (KS distance to the Gaussian of the same mean and std; skewness and excess kurtosis 0) and with the two Haar laws. Saved as `convergence_vs_n_samples.png` and `.json`; the histogram figure is saved with the shape table.

In [ ]:
from QELM.evaluation.feature_distribution import (
    convergence_table,
    distribution_moments,
    haar_moments,
    ks_to_normal,
)

convergence = {
    h: convergence_table(
        {r["n_samples"]: r["obs"]["Z"][:, SITE] for r in runs if r["h"] == h}
    )
    for h in hs
}
field_colors = color_map(f"h = {h:g} J" for h in hs)
fig_c, axes_c = subplots(1, 3, figsize=(19.5, 5))
for h in hs:
    rows_c, color = convergence[h], field_colors[f"h = {h:g} J"]
    n = np.array([row["n_samples"] for row in rows_c], dtype=float)
    smaller = n < n.max()
    axes_c[0].plot(
        n[smaller],
        [row["ks_to_reference"] for row in rows_c if row["n_samples"] < n.max()],
        "o-",
        color=color,
        label=f"h = {h:g} J",
    )
    axes_c[1].errorbar(
        n,
        [row["mean"] for row in rows_c],
        yerr=[row["sem"] for row in rows_c],
        fmt="o-",
        color=color,
        capsize=3,
        label=f"h = {h:g} J",
    )
    axes_c[2].plot(
        n, [row["std"] for row in rows_c], "o-", color=color, label=f"h = {h:g} J"
    )
# The scale of the sampling noise depends on the sizes only, not on h.
rows_c = convergence[hs[0]][:-1]
axes_c[0].plot(
    [row["n_samples"] for row in rows_c],
    [row["ks_bound"] for row in rows_c],
    "k--",
    lw=1,
    label="sqrt((n + m) / (n m))",
)
haar_par = haar_moments(D_parity)
haar_chg = haar_moments(D_charge, plus_charge)
for ax, key in ((axes_c[1], "mean"), (axes_c[2], "std")):
    ax.axhline(haar_par[key], color="k", ls="--", lw=1, label="Haar, parity sector")
    ax.axhline(haar_chg[key], color="k", ls=":", lw=1, label="Haar, charge sector")
axes_c[0].set(
    xscale="log",
    yscale="log",
    xlabel="n_samples",
    ylabel=f"KS distance to n = {sizes[-1]}",
    title=f"convergence of the law of <Z_{SITE + 1}>",
)
axes_c[1].set(xscale="log", xlabel="n_samples", ylabel="mean", title="mean")
axes_c[2].set(xscale="log", xlabel="n_samples", ylabel="std", title="std")
for ax in axes_c:
    ax.legend(fontsize=9)
fig_c.suptitle(f"Unitary QELM, N = {N}: law of large numbers for <Z_{SITE + 1}>")

shape_table = []
for h in hs:
    largest = next(r for r in runs if r["h"] == h and r["n_samples"] == sizes[-1])
    for site in range(Ns):
        values = largest["obs"]["Z"][:, site]
        shape_table.append(
            {
                "h": h,
                "site": site,
                **distribution_moments(values),
                "ks_to_normal": ks_to_normal(values),
            }
        )
references = {"Haar, parity sector": haar_par, "Haar, charge sector": haar_chg}

print(f"KS distance of <Z_{SITE + 1}> to the largest dataset (n = {sizes[-1]}):")
for h in hs:
    print(
        f"  h = {h:g} J: "
        + ", ".join(
            f"n = {row['n_samples']}: {row['ks_to_reference']:.3f} "
            f"(scale {row['ks_bound']:.3f})"
            for row in convergence[h][:-1]
        )
    )
print(f"\nShape of <Z_k> at n = {sizes[-1]}:")
print(f"{'':>22} {'mean':>7} {'std':>6} {'skew':>6} {'ex.kurt':>8} {'KS Gauss':>9}")
for row in shape_table:
    print(
        f"h = {row['h']:>4g} J, qubit {row['site']:>2}  {row['mean']:>+7.3f} "
        f"{row['std']:>6.3f} {row['skewness']:>+6.2f} {row['excess_kurtosis']:>+8.2f} "
        f"{row['ks_to_normal']:>9.3f}"
    )
for name, ref in references.items():
    print(
        f"{name:>22} {ref['mean']:>+7.3f} {ref['std']:>6.3f} "
        f"{ref['skewness']:>+6.2f} {ref['excess_kurtosis']:>+8.2f}"
    )

histogram_path = save_figure(
    fig_h,
    sweep_dir,
    "histograms_linear_observables",
    data={"shape": shape_table, "references": references, "site": SITE},
)
convergence_path = save_figure(
    fig_c,
    sweep_dir,
    "convergence_vs_n_samples",
    data={str(h): rows for h, rows in convergence.items()},
)
print(f"\nfigures: {histogram_path}, {convergence_path}")
plt.show()

## Analysis

Numbers of the default run (`N = 6`, so `Ns = 5`, `Q = 2`; `SITE = 2`, chain site 3; seeds 42 to 45; sweep `outputs/QELM_Z_linear/2026-10-25_00-21-34/`). They change a little with the seeds; the conclusions below do not.

**`<X_k>` and `<Y_k>` carry nothing.** Over all 2 x 7600 samples, `max |<X_k>| = max |<Y_k>| = 0` exactly, at both fields, as the parity argument requires. Their histograms are a single bar at 0 for every `n_samples`. The ridge readout on `<Y>` or `<X>` alone sees only the bias and predicts the mean of the train set: R2 between -0.07 (n = 100) and 0.00 (n = 5000). Of the `3 Ns` linear features, only the `Ns` values `<Z_k>` are informative; `<X_k>` and `<Y_k>` only become useful once the parity symmetry is broken (another initial state, or a term with an odd number of X or Y in `H`).

**Baseline of the linear readout.** On `<Z_k>`, alone or with the zeros, R2 = 1.0000 at `h = 10 J` for every `n_samples`: when the charge `sum_i (1 - Z_i) / 2` is nearly conserved, the ancilla `<Z_0>` is fixed by the sum of the system `<Z_k>`, an affine function the ridge finds with 100 samples. At `h = J`, R2 falls to 0.62 to 0.73: the couplings move the charge, and `<Z_0>` is no longer a linear function of the `<Z_k>`. More samples do not improve it (0.73, 0.69, 0.66, 0.62 for n = 100 to 5000, with test sets of 30 to 1500 samples, so the first values are the noisiest): the limit is the linearity of the readout on these features, not the size of the dataset.

**Convergence (law of large numbers).** The KS distance of `<Z_3>` to the largest dataset follows the sampling scale `sqrt((n + m) / (n m))`: 0.053, 0.039, 0.020 at `h = 10 J` and 0.099, 0.026, 0.029 at `h = J`, for scales 0.101, 0.047, 0.026 (n = 100, 500, 2000). Every distance is below the 5 % critical value `1.36 x scale`, so the independent datasets sample one law and the empirical distribution converges to it at the `1 / sqrt(n)` rate; the KDEs of `n = 2000` and `n = 5000` lie on top of each other. The mean is stable within its standard error from `n = 500` on; at `n = 100` the histogram is still ragged (two isolated bars at `h = J`) and the standard deviation is underestimated by about 10 %.

**Converged shape.** At `n = 5000` every system qubit has the same law (translation invariance of the disorder on average): at `h = 10 J`, mean 0.331 to 0.341, std 0.235 to 0.241, skewness -0.29 to -0.36, excess kurtosis -0.16 to 0.00.

- It is **not uniform** (excess kurtosis near 0, not -1.2, and all values in `[-0.68, 0.94]`, never near -1).
- It is **not centered and not symmetric**: the mean is +1/3 and the left tail is longer (skewness about -0.33). A Gaussian of the same mean and std is close (KS distance 0.024 to 0.034) but misses this asymmetry, which is significant: the standard error of the skewness at `n = 5000` is about `sqrt(6 / n) = 0.035`.
- It is **not the Haar law of the full symmetry sector**: a fully scrambled state in the parity sector (`D = 32`) would give mean 0, std 0.174, skewness 0 (dashed line, far off).
- It **is the Haar law of the charge sector**: a random state among the `C(6, 2) = 15` configurations of `Q = 2` excitations gives `<Z_k> = 2 B - 1` with `B ~ Beta(10, 5)`, of mean `1 - 2 Q / N = 1/3`, std 0.236, skewness -0.33, excess kurtosis -0.18 (dotted line). The measured moments match all four, and the dotted curve lies on the KDE of `n = 5000`. The structural asymmetry is therefore the charge conservation: at `h = 10 J` the transverse field nearly conserves `Q` (the `X_i X_j` couplings that change it by 2 are off-resonant by `4 h`), the dynamics scrambles the state inside the charge sector only, and the sector, with 2 excitations on 6 sites, favours `Z = +1`.

At `h = J` the charge is no longer conserved, yet the law hardly moves toward the parity-sector Haar law: mean 0.30, std 0.229, skewness -0.32 to -0.41. Our reading, not tested here: the unitary still conserves the energy, and with `h = J` the initial state remains closer in energy to the low-charge configurations, so the long-time state keeps a positive `<Z_k>`; the evolution time `t = 10 / J` may also be too short for a full scrambling. Sweeping `t` or comparing with the diagonal ensemble of `H` would separate the two.

**Consequence for the QELM.** The features fed to the classical readout are, at large field, the single-site marginals of a random state of a fixed charge sector: `Ns` correlated values of law `2 Beta(10, 5) - 1` constrained by `sum_k <Z_k>`, a geometry the linear readout exploits perfectly to recover the ancilla. The informative directions are set by the conserved quantities of the reservoir, and the parity symmetry removes two thirds of the linear features altogether.